### 1. Install OpenCV

In [2]:
!apt-get update -y && apt-get install -y libgl1 libglib2.0-0
!pip install --upgrade pip
!pip install ultralytics opencv-python-headless

Reading package lists... Done
E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)


In [2]:
import os, ultralytics
yaml_dir = os.path.join(os.path.dirname(ultralytics.__file__), "cfg", "models")
print("Looking into:", yaml_dir)
!ls $yaml_dir

Looking into: /home/shangche45/.local/lib/python3.12/site-packages/ultralytics/cfg/models
11  12	rt-detr  v10  v3  v5  v6  v8  v9


### 2. Data Preparation 

In [2]:
REVERSE_GAMMA   = 0.7
REVERSE_TARGET  = 3000.0

import os, math, torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from ultralytics import YOLO
from ultralytics.utils import LOGGER
from ultralytics.data.utils import check_det_dataset

#### label to meet YOLO form (只有第一次需要執行，以後不用)

In [ ]:
# SAFE convert: CSV pixel labels -> YOLO normalized labels (0..1), without touching originals
import os
from PIL import Image
from ultralytics.data.utils import check_det_dataset

def _read_lines(path):
    with open(path, 'r', newline='') as f:
        raw = f.read()
    raw = raw.replace('\r\n', '\n').replace('\r', '\n').lstrip('\ufeff')
    return [ln.strip() for ln in raw.split('\n') if ln.strip()]

def _write_lines(path, lines):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', newline='\n') as f:
        f.write('\n'.join(lines) + '\n')

def _img_for_stem(stem, images_root):
    # try common extensions
    for ext in ('.png', '.jpg', '.jpeg'):
        p = os.path.join(images_root, stem + ext)
        if os.path.exists(p):
            return p
    return None

def convert_labels_dir(images_root, labels_src, labels_dst):
    os.makedirs(labels_dst, exist_ok=True)
    converted, copied = 0, 0
    for fn in os.listdir(labels_src):
        if not fn.endswith('.txt'):
            continue
        stem = os.path.splitext(fn)[0]
        src = os.path.join(labels_src, fn)
        dst = os.path.join(labels_dst, fn)
        lines = _read_lines(src)
        if not lines:
            # empty -> copy as-is to keep dataset shape
            _write_lines(dst, [])
            copied += 1
            continue

        csv_like = sum(1 for ln in lines if (',' in ln and len(ln.split(',')) == 5))
        if csv_like == 0:
            _write_lines(dst, lines)
            copied += 1
            continue

        img_path = _img_for_stem(stem, images_root)
        if not img_path:
            _write_lines(dst, lines)
            copied += 1
            continue

        with Image.open(img_path) as im:
            W, H = im.size

        out = []
        for ln in lines:
            if ',' not in ln:
                continue
            parts = [p.strip() for p in ln.split(',')]
            if len(parts) != 5:
                continue
            c, xl, yt, ww, hh = parts
            try:
                c = int(c); xl = float(xl); yt = float(yt); ww = float(ww); hh = float(hh)
            except ValueError:
                continue
            xc = xl + ww/2.0
            yc = yt + hh/2.0
            xcn = max(0.0, min(1.0, xc / W))
            ycn = max(0.0, min(1.0, yc / H))
            wnn = max(0.0, min(1.0, ww / W))
            hnn = max(0.0, min(1.0, hh / H))
            out.append(f"{c} {xcn:.6f} {ycn:.6f} {wnn:.6f} {hnn:.6f}")

        _write_lines(dst, out if out else lines)
        converted += 1
    return converted, copied

# 1) read data.yaml
dd = check_det_dataset('data.yaml')
data_dict = dd[1] if isinstance(dd, (list, tuple)) and len(dd) > 1 else dd
train_img = data_dict['train']                         # data/images/train
val_img   = data_dict.get('val', '')                   # data/images/val

train_lbl_src = train_img.replace(os.sep+'images'+os.sep+'train', os.sep+'labels'+os.sep+'train')
val_lbl_src   = val_img.replace(os.sep+'images'+os.sep+'val',   os.sep+'labels'+os.sep+'val')

train_lbl_dst = train_img.replace(os.sep+'images'+os.sep+'train', os.sep+'labels_yolo'+os.sep+'train')
val_lbl_dst   = val_img.replace(os.sep+'images'+os.sep+'val',   os.sep+'labels_yolo'+os.sep+'val')

# 2) transform
c1, k1 = convert_labels_dir(train_img, train_lbl_src, train_lbl_dst)
c2, k2 = convert_labels_dir(val_img,   val_lbl_src,   val_lbl_dst)
print(f"[SAFE CONVERT] train converted={c1}, copied={k1}  |  val converted={c2}, copied={k2}")

# 3) switch
import shutil
def safe_swap(src_dir, dst_dir, bak_dir):
    if os.path.exists(dst_dir) and not os.path.exists(bak_dir):
        shutil.move(dst_dir, bak_dir)     # labels  -> labels_bak
    if os.path.exists(src_dir):
        shutil.move(src_dir, dst_dir)     # labels_yolo -> labels

train_lbl = train_lbl_src                      # data/labels/train
val_lbl   = val_lbl_src                        # data/labels/val
train_lbl_yolo = train_lbl_dst                 # data/labels_yolo/train
val_lbl_yolo   = val_lbl_dst                   # data/labels_yolo/val
train_lbl_bak  = train_lbl.replace('/labels/', '/labels_bak/')
val_lbl_bak    = val_lbl.replace('/labels/',   '/labels_bak/')

os.makedirs(os.path.dirname(train_lbl_bak), exist_ok=True)
os.makedirs(os.path.dirname(val_lbl_bak),   exist_ok=True)

safe_swap(os.path.dirname(train_lbl_yolo), os.path.dirname(train_lbl), os.path.dirname(train_lbl_bak))
safe_swap(os.path.dirname(val_lbl_yolo),   os.path.dirname(val_lbl),   os.path.dirname(val_lbl_bak))

print("[SAFE CONVERT] Swapped labels_yolo -> labels (original saved in labels_bak).")

[SAFE CONVERT] train converted=0, copied=760  |  val converted=0, copied=190
[SAFE CONVERT] Swapped labels_yolo -> labels (original saved in labels_bak).


### 3. Read `.yaml`

In [4]:
import os, math
from typing import Dict, List, Tuple

def cosine_alpha(epoch: int, total_epochs: int) -> float:
    return 0.5 * (1.0 + math.cos(math.pi * epoch / max(1, total_epochs)))

def compute_class_image_stats(labels_dir: str, names: Dict[int, str]) -> Tuple[List[int], List[int]]:
    num_classes = len(names)
    inst = [0] * num_classes
    img_has = [0] * num_classes
    if not os.path.isdir(labels_dir):
        return inst, img_has

    for fn in os.listdir(labels_dir):
        if not fn.endswith('.txt'):
            continue
        present = set()
        with open(os.path.join(labels_dir, fn), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                try:
                    c = int(parts[0])
                except ValueError:
                    continue
                if 0 <= c < num_classes:
                    inst[c] += 1
                    present.add(c)
        for c in present:
            img_has[c] += 1
    return inst, img_has

def effective_num_weights(class_counts: List[int], beta: float = 0.999) -> List[float]:

    ws = []
    for n in class_counts:
        if n <= 0:
            ws.append(0.0)
        else:
            w = (1.0 - beta) / (1.0 - beta ** n)
            ws.append(w)
    mean_w = sum(ws) / max(1, len(ws))
    return [w / (mean_w if mean_w > 0 else 1.0) for w in ws]

def image_repeat_factors(labels_dir: str, names: Dict[int, str], target: float, gamma: float) -> Dict[str, float]:

    num_classes = len(names)
    inst, _ = compute_class_image_stats(labels_dir, names)

    if not target or target <= 0:
        nz = sorted([n for n in inst if n > 0])
        target = float(nz[len(nz)//2]) if nz else 1.0

    class_factor = [1.0] * num_classes
    for c in range(num_classes):
        n = inst[c]
        if n > 0:
            class_factor[c] = max(1.0, (target / float(n)) ** gamma)

    repeat = {}
    if not os.path.isdir(labels_dir):
        return repeat

    for fn in os.listdir(labels_dir):
        if not fn.endswith('.txt'):
            continue
        path = os.path.join(labels_dir, fn)
        factors = [1.0]
        with open(path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                try:
                    c = int(parts[0])
                except ValueError:
                    continue
                if 0 <= c < num_classes:
                    factors.append(class_factor[c])
        repeat[fn[:-4]] = max(factors) 
    return repeat

In [5]:
# 1) Read data.yaml
data_dict   = check_det_dataset("data.yaml") 
train_img   = data_dict["train"]
names       = data_dict["names"]
labels_train = train_img.replace(os.sep+'images'+os.sep+'train',
                                 os.sep+'labels'+os.sep+'train')

class_inst, _ = compute_class_image_stats(labels_train, names)
reverse_w_map = image_repeat_factors(labels_train, names, REVERSE_TARGET, REVERSE_GAMMA)

In [6]:
# 2) Define callback and some settings for BBN-lite
def on_train_start(trainer):
    trainer.args.model         = 'yolov8x.yaml' 
    trainer.args.pretrained    = False
    trainer.args.imgsz         = 1536
    trainer.args.optimizer     = 'adamw'
    trainer.args.lr0           = 0.002
    trainer.args.weight_decay  = 0.05
    trainer.args.warmup_epochs = 3
    trainer._alpha             = 1.0
    trainer.args.close_mosaic = 0 #new
    print("[BBN] REVERSE_GAMMA =", REVERSE_GAMMA, "REVERSE_TARGET =", REVERSE_TARGET)

def _rebuild_weighted_loader(trainer, alpha):
    ds = trainer.train_loader.dataset
    weights = []
    for im_path in getattr(ds, 'im_files', []):
        key = os.path.basename(im_path).rsplit('.', 1)[0]
        u, r = 1.0, reverse_w_map.get(key, 1.0)
        eff  = alpha * u + (1.0 - alpha) * r
        weights.append(float(max(1e-6, eff)))
    if not weights:
        return
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
    trainer.train_loader = DataLoader(ds, batch_size=trainer.args.batch,
                                      sampler=sampler, num_workers=trainer.args.workers,
                                      pin_memory=True, collate_fn=ds.collate_fn)

def on_train_epoch_start(trainer):
    epoch, total = trainer.epoch, trainer.args.epochs

    # 1) alpha modification
    alpha = 0.5 * (1.0 + math.cos(math.pi * epoch / max(1, total)))
    trainer._alpha = alpha

    # 2) WeightedRandomSampler
    try:
        _rebuild_weighted_loader(trainer, alpha)
    except Exception as e:
        LOGGER.warning(f"[BBN] Rebuild loader failed: {e}")

    # 3) Log
    LOGGER.info(f"[BBN][epoch {epoch+1}/{total}] α={trainer._alpha:.3f} | Stage: Uniform → Reverse")

### 4. Training

In [ ]:
#training
from ultralytics import YOLO

model = YOLO('yolov8x.yaml')
model.add_callback('on_train_start', on_train_start)
model.add_callback('on_train_epoch_start', on_train_epoch_start) 

results = model.train(
    data="data.yaml",
    epochs=150,
    imgsz=1280,
    batch=8,
    device=0,
    pretrained=False,
    optimizer="adamw",
    lr0=0.002,     
    lrf=0.01,       
    weight_decay=0.01,
    cache="ram",
    workers=2,
    patience=50, 
    amp=True,
    cos_lr=True,
    mosaic=0.0, 
    close_mosaic=0,
)

New https://pypi.org/project/ultralytics/8.3.221 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.217 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (Tesla V100-SXM2-32GB, 32501MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1536, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.05, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8x.yaml, momentum=0.937, mosaic=0.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



     72/150      11.1G      2.049      1.089      1.388         14       1536: 100% ━━━━━━━━━━━━ 380/380 3.2it/s 1:59<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 7.8it/s 6.2s0.1s
                   all        190       6390      0.672      0.546      0.569       0.22
[BBN][epoch 73/150] α=0.531 | Stage: Uniform → Reverse

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     73/150      11.1G      2.031      1.104      1.395         35       1536: 100% ━━━━━━━━━━━━ 380/380 3.2it/s 1:59<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 7.7it/s 6.2s0.1s
                   all        190       6390      0.718      0.538      0.579      0.226
[BBN][epoch 74/150] α=0.521 | Stage: Uniform → Reverse

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     74/150      11.1G      2.039      1.094 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



    107/150      11.2G      1.905     0.9768      1.341         93       1536: 100% ━━━━━━━━━━━━ 380/380 3.2it/s 2:00<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 7.7it/s 6.2s0.1s
                   all        190       6390      0.676      0.631      0.629      0.245
[BBN][epoch 108/150] α=0.189 | Stage: Uniform → Reverse

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    108/150      11.2G      1.948     0.9903      1.331         46       1536: 100% ━━━━━━━━━━━━ 380/380 3.2it/s 2:00<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 7.6it/s 6.3s0.1s
                   all        190       6390      0.669      0.646      0.616       0.24
[BBN][epoch 109/150] α=0.181 | Stage: Uniform → Reverse

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    109/150      11.2G      1.921     0.9927

### 5. Testing

In [11]:
#predict
from ultralytics import YOLO

model = YOLO('/home/shangche45/hw2_NTHU_114064545/runs/detect/train/weights/best.pt')  # 你訓練好的權重
results = model.predict(
    source="/home/shangche45/hw2_NTHU_114064545/data/images/test",
    imgsz=1536,
    conf=0.0005,
    max_det=1200,
    augment=True,
    save=True,
    save_txt=True,
    save_conf=True,
    project="runs/predict",
    name="bbn_y8n"
)


image 1/550 /home/shangche45/hw2_NTHU_114064545/data/images/test/img0001.png: 864x1536 233 cars, 20 hovs, 446 persons, 331 motorcycles, 631.6ms
image 2/550 /home/shangche45/hw2_NTHU_114064545/data/images/test/img0002.png: 864x1536 717 cars, 83 hovs, 73 persons, 327 motorcycles, 133.9ms
image 3/550 /home/shangche45/hw2_NTHU_114064545/data/images/test/img0003.png: 832x1536 39 cars, 3 hovs, 1146 persons, 12 motorcycles, 125.0ms
image 4/550 /home/shangche45/hw2_NTHU_114064545/data/images/test/img0004.png: 832x1536 38 cars, 7 hovs, 1098 persons, 57 motorcycles, 122.1ms
image 5/550 /home/shangche45/hw2_NTHU_114064545/data/images/test/img0005.png: 832x1536 108 cars, 35 hovs, 984 persons, 73 motorcycles, 122.4ms
image 6/550 /home/shangche45/hw2_NTHU_114064545/data/images/test/img0006.png: 832x1536 101 cars, 27 hovs, 996 persons, 76 motorcycles, 120.9ms
image 7/550 /home/shangche45/hw2_NTHU_114064545/data/images/test/img0007.png: 864x1536 78 cars, 14 hovs, 328 persons, 426 motorcycles, 126.3ms

### 6. Output

In [12]:
#generate submission file
!python /home/shangche45/hw2_NTHU_114064545/make_submission.py

Wrote /home/shangche45/hw2_NTHU_114064545/submission.csv with 550 rows.
